# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² mass spectrometry dataset of human mast cell extracellular vesicles using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

In [ ]:
# List record sets with their @id
print('Available RecordSets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# List fields and columns for each record set
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}")
    print('  Fields:')
    for f in rs.get('field', []):
        if isinstance(f, dict):
            print(f"    Field @id: {f.get('@id')} | name: {f.get('name', 'N/A')} | dataType: {f.get('dataType', 'N/A')}")
        else:
            print(f"    {f}")
    # List columns in attached FileObjects, if present
    file_objs = rs.get('fileObject')
    if file_objs:
        if not isinstance(file_objs, list):
            file_objs = [file_objs]
        for fo in file_objs:
            columns = fo.get('column') if isinstance(fo, dict) else None
            if columns:
                print('  Columns:')
                for c in columns:
                    if isinstance(c, dict):
                        print(f"    Column @id: {c.get('@id')} | name: {c.get('name', 'N/A')}")
                    else:
                        print(f"    {c}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Note: Replace the values below with the `@id`s of the record sets you want to extract. You can select multiple or one record set to proceed with analysis. For demonstration, we use the main record set you inspect above._

In [ ]:
# List all record set @ids (from overview above)
record_set_ids = [rs["@id"] for rs in dataset.record_sets]
print('Record set @ids:', record_set_ids)

# --- Choose main data record set (by @id) ---
# As of Croissant 1.0, main tabular record set typically has the form <dataset_url>#main or similar.
# Set the 'main_records_set_id' to a record set of interest from above. Adjust as necessary based on displayed options.
main_record_set_id = record_set_ids[0]  # Replace or loop for multiple, if desired.

# Load the records for all available record sets into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows for RecordSet {rs_id}")
        else:
            print(f"No records found for RecordSet {rs_id}")
    except Exception as e:
        print(f"Skipping RecordSet {rs_id} due to error: {e}")

# List columns for main record set
if main_record_set_id in dataframes and not dataframes[main_record_set_id].empty:
    print('Columns:', dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('Main RecordSet DataFrame is empty.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

_Please change the field names/IDs below to those relevant fields from your loaded columns list. Example fields from a typical mass spec protein dataset might include: a count, coverage, abundance, MW, or other numeric metrics._

In [ ]:
# Set your record set and select a numeric and a grouping field (adjust these after inspecting your DataFrame columns above)
record_set_id = main_record_set_id
df = dataframes[record_set_id]
# Choose a numeric field from df.columns for filtering/normalization. For illustration, try to guess a reasonable field:
numeric_candidates = [col for col in df.columns if any(x in col.lower() for x in ['count', 'abundance', 'coverage', 'mw', 'intensity'])]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    numeric_field = df.columns[0]
print(f"Using numeric field: {numeric_field}")

# Filter rows above a threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
else:
    print(f"Field '{numeric_field}' is not numeric type; skipping filtering and normalization.")

# Try grouping by a possible categorical field
group_candidates = [col for col in df.columns if any(x in col.lower() for x in ['type', 'class', 'group', 'category', 'sample']) and col != numeric_field]
if group_candidates and pd.api.types.is_numeric_dtype(df[numeric_field]):
    group_field = group_candidates[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    display(grouped_df.head())
else:
    print('No group field found for grouping.' if not group_candidates else 'Numeric field is not suitable for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

_Below we plot a histogram for the selected numeric field, and (if there is a grouping variable) a barplot of group means._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print(f"Field '{numeric_field}' is not numeric type; skipping histogram.")

# If group_field exists, show group means as barplot
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10,4))
    sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
    plt.xticks(rotation=45)
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the [FAIR² mass spectrometry dataset of extracellular vesicle proteins from human mast cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using `mlcroissant`. We previewed the metadata, examined available record sets and fields via their `@id`, and performed basic exploratory analysis and visualization on one record set. For comprehensive downstream research (e.g., proteomic biomarker detection or comparative analyses), further feature engineering and domain-specific filtering may be required.

For more advanced usage, consult the [`mlcroissant`](https://mlcommons.github.io/croissant/) documentation and explore additional fields and annotation within the Croissant schema.